In [2]:
import numpy as np
import pandas as pd

df = pd.read_excel('Final_Financial_Dataset_Practical.xlsx')

In [ ]:
#1. Create a list of all the desired column names

feature_cols = ['units sold', 'mau/aau', 'churn rate', 'funding_rounds', 'seed', 'venture']

X_features = df[feature_cols]

y_net_revenue = df['beginning revenue'] + df['month_1_revenue'] + df['month_2_revenue'] + df['month_3_revenue'] + df['month_4_revenue'] + df['month_5_revenue'] + df['month_6_revenue'] - (df['operating expenses'] + df['COGS'] + df['Discounts'] + df['Returns & Allowances'])


In [4]:
df.head()

,name,homepage_url,category_list,market,funding_total_usd,status,country_code,state_code,region,city,...,month_4_revenue,month_5_revenue,month_6_revenue,Operating Expense,COGS,Discounts,Units Sold,MAU/AAU,Churn Rate,Expansion Revenue
0,#waywire,http://www.waywire.com,|Entertainment|Politics|Social Media|News|,News,1750000,acquired,USA,NY,New York City,New York,...,7.377459e+06,6.616146e+06,5.933349e+06,1.742239e+06,2.839177e+06,458052.098548,11976,271413,8.540925,7.382048e+06
1,&TV Communications,http://enjoyandtv.com,|Games|,Games,4000000,operating,USA,CA,Los Angeles,Los Angeles,...,4.090864e+06,3.502497e+06,3.996254e+06,1.749075e+06,1.894599e+06,401796.318229,49640,178678,5.798359,3.742446e+06
2,'Rock' Your Paper,http://www.rockyourpaper.org,|Publishing|Education|,Publishing,40000,operating,EST,NaN,Tallinn,Tallinn,...,1.478405e+06,1.391582e+06,1.401918e+06,5.369323e+05,9.419153e+05,134606.643285,34586,441856,3.142732,1.320542e+06
3,(In)Touch Network,http://www.InTouchNetwork.com,|Electronics|Guides|Coffee|Restaurants|Music|i...,Electronics,1500000,operating,GBR,NaN,London,London,...,2.944374e+06,2.760773e+06,2.650090e+06,1.396144e+06,1.258432e+06,93786.649485,5868,390485,9.976369,2.567490e+06
4,-R- Ranch and Mine,NaN,|Tourism|Entertainment|Games|,Tourism,60000,operating,USA,TX,Dallas,Fort Worth,...,4.558715e+05,4.685103e+05,4.047625e+05,1.457553e+05,1.949275e+05,14136.307787,17396,54135,9.828168,6.311658e+05


In [25]:
X_train = X_features.to_numpy()

In [5]:
#step 1 : Initial Prediction - Start with a simple base prediction(the average of y_net_revenue)

F0 = sum(y_net_revenue)/ len(y_net_revenue)
#current prediction

In [6]:
#step 2 : Calculate Residuals(Errors) - Find the diffrence between actual y and current prediction

# Residual = y(actual) - y(prediction)
Residual = y_net_revenue - F0 

In [7]:
!pip install tensorflow

In [8]:
#step 3. Train a weak learner - Train a small neural network(h1) to predict the residuals.

# The Neural Network Blueprint

# 1. Input Layer - Needs to accept 6 features ( X_features.shape[1])
# 2. Hidden Layer - We'll use a small layer (e.g 8 neurons) with a ReLU activation function. This keeps the model "weak" and focused.
# 3. Output Layer - Needs to output a single number( the predicted residual). The activation here is typically linear for regression task.
# 4. Compilation - We must define the optimizer (e.g 'adam') and the loss function. Since residuals are continuous numbers, we use Mean Squared Error('mse') as the loss function.


from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

def create_weak_learner(input_dim):
    model = Sequential([
        Dense(8, activation='relu', input_shape=(input_dim,)),

        Dense(1)
    ])

    model.compile(optimizer='adam', loss='mse')

    # Compile the model
    model.compile(optimizer='adam', loss='mse')
    return model


In [9]:
# Instantiate the model

h1 = create_weak_learner(6)

#Train h1 to predict the residual(y_train) from feature(X_train)
history = h1.fit(
    X_features,
    Residual,
    epochs=20,      #Use a small no. of epochs to keep model weak
    verbose=0       #Suppress output for clean execution
)

print(f"Training of h1 complete: Final MSE Loss {history.history['loss'][-1]:,.2f}")


C:\Users\PRATHMESH\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training of h1 complete: Final MSE Loss 642,873,120,784,384.00


In [10]:
# step 4: Update prediction - Add the prediction of the new learner to the initial prediction
#  y(new prediction) = y(initial prediction) + h1(X)


#Get the y_predicted from h1
#flatten() is used to convert [[p1], [p2]] to [p1, p2]

h1_predictions = h1.predict(X_features).flatten() # h1(X)

F1 = F0 + h1_predictions



63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step  


In [11]:
print("First 5 initial predictions (F0):")
print(F0)


First 5 initial predictions (F0):
10492732.533570714


In [12]:
print("\nFirst 5 h1 predictions (h1's guess for residuals):")
print(h1_predictions[:5])


First 5 h1 predictions (h1's guess for residuals):
[-1.6452508e+06 -4.1955716e+05 -9.7609244e+05  3.4500155e+06
 -5.8819931e+05]


In [13]:
print("\nFirst 5 new combined predictions (F1):")
print(F1[:5])




First 5 new combined predictions (F1):
[8.8474820e+06 1.0073176e+07 9.5166410e+06 1.3942748e+07 9.9045340e+06]


In [15]:
!pip install tabulate

In [16]:
#Calculate second set of residuals
r2 = y_net_revenue - F1

y_train = r2.to_numpy().astype(np.float32)

print("First 5 new residuals (r2) head: ")
print(r2.head().to_markdown(index=False, numalign='left', stralign='left'))


First 5 new residuals (r2) head: 
| 0            |
|:-------------|
| 3.20899e+07  |
| 2.78694e+07  |
| -3.01208e+07 |
| 9.90582e+06  |
| -3.14201e+07 |


In [21]:
#Create the second weak learner
h2 = create_weak_learner(X_features.shape[1])

print("Training h2: ")
history = h2.fit(
    X_features, 
    r2,
    epochs=20,
    verbose=0
)

print(f"Training of h2 is complete: Final MSE loss: {history.history['loss'][-1]}")

C:\Users\PRATHMESH\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training h2: 
Training of h2 is complete: Final MSE loss: 673436040953856.0


In [22]:
h2_predictions = h2.predict(X_features).flatten()

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  


In [23]:
F2 = F1 + h2_predictions

In [24]:
print("Comparisons of of prediction improvements: ")
print(f"Actual net revenue sample: {y_net_revenue[:5].to_numpy().round(0)}")

print(f"Prediction Round 1 (F1) Sample: {F1[:5].round(0)}")

print(f"Prediction Round 2 (F2) Sample: {F2[:5].round(0)}")

Comparisons of of prediction improvements: 
Actual net revenue sample: [ 40937369.  37942572. -20604119.  23848567. -21515546.]
Prediction Round 1 (F1) Sample: [8.8474820e+06 1.0073176e+07 9.5166410e+06 1.3942748e+07 9.9045340e+06]
Prediction Round 2 (F2) Sample: [9.7699420e+06 1.1415495e+07 9.0453920e+06 1.2305327e+07 1.2686449e+07]


In [26]:
M = 10 #number of boosting rounds(i.e decision trees)
initial_prediction = y_net_revenue.mean()
current_prediction = initial_prediction.copy()
weak_leaners = [] #List to store all models/weak learners(h1, h2, h3, .....)


for m in range(M):
    #Calculate residuals
    residual = y_net_revenue - current_prediction


    #train a new weak learner
    h_new = create_weak_learner(X_features.shape[1])
    h_new.fit(
        X_train, residual.to_numpy().astype(np.float32), epochs=20, verbose=0)
    weak_leaners.append(h_new)

    #get prediction
    h_prediction = h_new.predict(X_train).flatten()

    #update prediction
    LEARNING_RATE = 0.1
    current_prediction = current_prediction + (LEARNING_RATE*h_prediction)



C:\Users\PRATHMESH\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 840us/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 793us/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 894us/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 934us/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [28]:
F_final = current_prediction
print("\nFinal Model Summary:")
print(f"Final Prediction (Net revenue): {np.mean(np.abs(y_net_revenue - F_final)):,.0f}")
print(f"Initial Prediction (Net revenue): {np.mean(np.abs(y_net_revenue - initial_prediction)):,.0f}")


Final Model Summary:
Final Prediction (Net revenue): 20,666,932
Initial Prediction (Net revenue): 20,692,351
